In [1]:
import pandas as pd

# Load the raw consumption data
consumption_path = '../../data/paraguay/electricity-consumption-raw.csv'
df_consumption = pd.read_csv(consumption_path)

In [2]:
meterological_path = '../../data/paraguay/meteorological-raw.csv'
df_meterological = pd.read_csv(meterological_path)

# Load substation location data
substation_path = '../../data/paraguay/substations-geographical-location.csv'
df_substations = pd.read_csv(substation_path)


In [3]:
from IPython.display import display, Markdown
def describe_dataframe(df, explanations, name):
    rows = []
    for col in df.columns:
        dtype = str(df[col].dtype)
        expl = explanations.get(col, "")
        # For substation code, force string and sort alphabetically for min/max
        if col.lower() == 'substation' or col.lower() == 'code':
            col_as_str = df[col].astype(str)
            min_val = col_as_str.min()
            max_val = col_as_str.max()
            mean_val = median_val = "-"
        elif pd.api.types.is_numeric_dtype(df[col]):
            min_val = df[col].min()
            max_val = df[col].max()
            mean_val = df[col].mean()
            median_val = df[col].median()
        elif pd.api.types.is_datetime64_any_dtype(df[col]) or (dtype.startswith('object') and 'time' in col):
            try:
                col_as_dt = pd.to_datetime(df[col], errors='coerce')
                min_val = col_as_dt.min()
                max_val = col_as_dt.max()
                mean_val = median_val = "-"
            except Exception:
                min_val = max_val = mean_val = median_val = "-"
        else:
            min_val = max_val = mean_val = median_val = "-"
        rows.append({
            'Column': col,
            'Datatype': dtype,
            'Explanation': expl,
            'Min': min_val,
            'Max': max_val,
            'Mean': mean_val,
            'Median': median_val
        })
    summary_df = pd.DataFrame(rows)
    display(Markdown(f"#### {name} column summary"))
    display(summary_df)

# Short explanations for each dataframe's columns
consumption_expl = {
    'datetime': 'Timestamp of measurement',
    'feeder': 'Unique identifier for the feeder (distribution line)',
    'substation': 'Code for the substation where the feeder is located',
    'consumption': 'Electricity consumption value'
}
meterological_expl = {
    'datetime': 'Timestamp of weather measurement',
    'temperature': 'Air temperature at the time of measurement (°C)',
    'humidity': 'Relative humidity (%)',
    'wind_speed': 'Wind speed (m/s)',
    'precipitation': 'Precipitation (mm)',
    'pressure': 'Atmospheric pressure (hPa)'
}
substations_expl = {
    'Code': 'Code for the substation',
    'Substation Name': 'Full name of the substation',
    'Latitude': 'Latitude coordinate of the substation',
    'Longitude': 'Longitude coordinate of the substation'
}

describe_dataframe(df_consumption, consumption_expl, "Consumption Data")
describe_dataframe(df_meterological, meterological_expl, "Meteorological Data")
describe_dataframe(df_substations, substations_expl, "Substation Data")

#### Consumption Data column summary

,Column,Datatype,Explanation,Min,Max,Mean,Median
0,datetime,object,Timestamp of measurement,2017-01-01 00:00:00,2020-12-31 23:00:00,-,-
1,substation,object,Code for the substation where the feeder is lo...,A,N,-,-
2,feeder,object,Unique identifier for the feeder (distribution...,-,-,-,-
3,consumption,float64,Electricity consumption value,0.7,2202.0,101.881147,89.0


#### Meteorological Data column summary

,Column,Datatype,Explanation,Min,Max,Mean,Median
0,datetime,object,Timestamp of weather measurement,2017-01-01 00:00:00,2020-12-31 23:00:00,-,-
1,temperature,float64,Air temperature at the time of measurement (°C),-0.2,40.8,23.028848,23.0
2,humidity,float64,Relative humidity (%),12.0,100.0,71.905786,75.0
3,wind_speed,float64,Wind speed (m/s),0.0,92.7,11.687591,10.0
4,pressure,float64,Atmospheric pressure (hPa),969.4,1003.5,984.817523,984.3


#### Substation Data column summary

,Column,Datatype,Explanation,Min,Max,Mean,Median
0,Code,object,Code for the substation,A,N,-,-
1,Substation Name,object,Full name of the substation,-,-,-,-
2,Latitude,float64,Latitude coordinate of the substation,-26.440487,-23.985915,-25.309727,-25.46848
3,Longitude,float64,Longitude coordinate of the substation,-55.682556,-54.386487,-54.86284,-54.740489


In [4]:
# Group the data by feeder and create a dictionary of dataframes
feeder_dfs = {}
for feeder, group in df_consumption.groupby('feeder'):
    feeder_dfs[feeder] = group[['datetime', 'feeder', 'substation', 'consumption']].reset_index(drop=True)

def print_dataframe(feeder_name, n=5):
    if feeder_name in feeder_dfs:
        display(feeder_dfs[feeder_name].head(n))
    else:
        print(f"Feeder '{feeder_name}' not found.")

In [5]:
# Load the weather data
weather_path = '../../data/paraguay/meteorological-raw.csv'
df_weather = pd.read_csv(weather_path)

# Ensure both datetime columns are in datetime format
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'])
for feeder in feeder_dfs:
    feeder_dfs[feeder]['datetime'] = pd.to_datetime(feeder_dfs[feeder]['datetime'])

# Merge weather data into each feeder dataframe on the datetime column
for feeder in feeder_dfs:
    feeder_dfs[feeder] = pd.merge(
        feeder_dfs[feeder],
        df_weather,
        how='left',
        on='datetime'
    )

In [6]:
# Load substation location data
substation_path = '../../data/paraguay/substations-geographical-location.csv'
df_substations = pd.read_csv(substation_path)

# Prepare substation dataframe: rename columns for merging
df_substations = df_substations.rename(columns={
    'Substation Name': 'substation_name',
    'Code': 'substation',
    'Latitude': 'latitude',
    'Longitude': 'longitude'
})

# Merge substation info into each feeder dataframe on 'substation'
for feeder in feeder_dfs:
    feeder_dfs[feeder] = pd.merge(
        feeder_dfs[feeder],
        df_substations[['substation', 'substation_name', 'latitude', 'longitude']],
        how='left',
        on='substation'
    )

In [17]:
import numpy as np
import plotly.express as px
from itertools import combinations
from geopy.distance import geodesic

# Calculate pairwise distances between substations
def calculate_station_distances(df):
    coords = df[['substation_name', 'latitude', 'longitude']].drop_duplicates()
    coords = coords.dropna()
    pairs = list(combinations(coords.itertuples(index=False), 2))
    distances = []
    for a, b in pairs:
        dist = geodesic((a.latitude, a.longitude), (b.latitude, b.longitude)).kilometers
        distances.append({'from': a.substation_name, 'to': b.substation_name, 'distance_km': dist})
    return distances

# Prepare substation coordinates for calculation
if 'substation_name' in df_substations.columns:
    coords_df = df_substations.rename(columns={
        'substation_name': 'substation_name',
        'substation': 'substation',
        'latitude': 'latitude',
        'longitude': 'longitude'
    })
else:
    coords_df = df_substations.rename(columns={
        'Substation Name': 'substation_name',
        'Code': 'substation',
        'Latitude': 'latitude',
        'Longitude': 'longitude'
    })

distances = calculate_station_distances(coords_df)
distances_df = pd.DataFrame(distances)
max_dist = distances_df['distance_km'].max()
avg_dist = distances_df['distance_km'].mean()

display(Markdown(f"**Maximum distance between substations:** {max_dist:.2f} km"))
display(Markdown(f"**Average distance between substations:** {avg_dist:.2f} km"))

# Extra info: Closest and farthest pairs
max_pair = distances_df.loc[distances_df['distance_km'].idxmax()]
min_pair = distances_df.loc[distances_df['distance_km'].idxmin()]
display(Markdown(f"**Farthest substations:** {max_pair['from']} ↔ {max_pair['to']} ({max_pair['distance_km']:.2f} km)"))
display(Markdown(f"**Closest substations:** {min_pair['from']} ↔ {min_pair['to']} ({min_pair['distance_km']:.2f} km)"))

# Map of substations with substation code as text label
fig = px.scatter_mapbox(
    coords_df,
    lat='latitude',
    lon='longitude',
    hover_name='substation_name',
    text='substation',  # Show the tag (A, B, C, etc) on the map
    zoom=7,
    height=700,
    width=700,
    mapbox_style='open-street-map'
 )
fig.update_traces(textposition='top center')
fig.show()

**Maximum distance between substations:** 273.90 km

**Average distance between substations:** 100.31 km

**Farthest substations:** Estación Carlos Antonio López ↔ Estación Salto del Guairá (273.90 km)

**Closest substations:** Subestación Acaray ↔ Subestación Hernandarias (2.44 km)

C:\Users\knuto\AppData\Local\Temp\ipykernel_84016\1529697127.py:48: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [24]:
# --- Correlation heatmap of substations: temperature, wind, humidity, pressure vs production ---
import pandas as pd
import plotly.express as px

# 1. Prepare daily production per substation
df = pd.concat(feeder_dfs.values(), ignore_index=True)
df['datetime'] = pd.to_datetime(df['datetime'])
df['date'] = df['datetime'].dt.date

# Sum production per substation per day
substation_daily = df.groupby(['substation', 'substation_name', 'latitude', 'longitude', 'date'])['consumption'].sum().reset_index()

# 2. Prepare daily weather per substation (average of all feeders at that substation)
weather_vars = ['temperature', 'wind_speed', 'humidity', 'pressure']
weather_daily = {}
for var in weather_vars:
    if var in df.columns:
        weather_daily[var] = df.groupby(['substation', 'date'])[var].mean().reset_index()
    else:
        weather_daily[var] = pd.DataFrame()

# 3. Merge production and weather variables
merged = substation_daily.copy()
for var in weather_vars:
    if not weather_daily[var].empty:
        merged = pd.merge(merged, weather_daily[var], on=['substation', 'date'], how='left', suffixes=('', f'_{var}'))

# 4. Calculate correlation for each substation between production and each weather variable
corrs = []
for sub, group in merged.groupby('substation'):
    result = {
        'substation': sub,
        'substation_name': group['substation_name'].iloc[0],
        'latitude': group['latitude'].iloc[0],
        'longitude': group['longitude'].iloc[0],
    }
    for var in weather_vars:
        if var in group.columns and group[var].notna().sum() > 2:
            result[f'corr_{var}'] = group['consumption'].corr(group[var])
        else:
            result[f'corr_{var}'] = None
    corrs.append(result)
corrs_df = pd.DataFrame(corrs)

# 5. Plot heatmaps for each weather variable
for var in weather_vars:
    if f'corr_{var}' in corrs_df.columns:
        fig = px.scatter_mapbox(
            corrs_df,
            lat='latitude',
            lon='longitude',
            hover_name='substation_name',
            text='substation',
            color=f'corr_{var}',
            color_continuous_scale='RdBu',
            range_color=[-1, 1],
            zoom=7,
            height=700,
            width=700,
            mapbox_style='open-street-map',
            title=f'Correlation between {var.replace("_", " ").title()} and Production per Substation'
        )
        fig.update_traces(textposition='top center', marker=dict(size=32))
        fig.show()

C:\Users\knuto\AppData\Local\Temp\ipykernel_84016\437142.py:48: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



C:\Users\knuto\AppData\Local\Temp\ipykernel_84016\437142.py:48: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



C:\Users\knuto\AppData\Local\Temp\ipykernel_84016\437142.py:48: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



C:\Users\knuto\AppData\Local\Temp\ipykernel_84016\437142.py:48: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [8]:
print_dataframe("C1")

,datetime,feeder,substation,consumption,temperature,humidity,wind_speed,pressure,substation_name,latitude,longitude
0,2017-01-01 00:00:00,C1,C,NaN,26.0,85.0,9.3,982.5,Estación Itakyry,-24.994788,-55.074876
1,2017-01-01 01:00:00,C1,C,92.0,NaN,NaN,NaN,NaN,Estación Itakyry,-24.994788,-55.074876
2,2017-01-01 02:00:00,C1,C,67.0,NaN,NaN,NaN,NaN,Estación Itakyry,-24.994788,-55.074876
3,2017-01-01 03:00:00,C1,C,85.0,25.0,94.0,7.4,981.8,Estación Itakyry,-24.994788,-55.074876
4,2017-01-01 04:00:00,C1,C,85.0,NaN,NaN,NaN,NaN,Estación Itakyry,-24.994788,-55.074876


In [33]:
print(f"number of feeders: {len(feeder_dfs)}")
print(f"number of substations: {len(df_substations)}")
# Calculate the number of unique feeders per substation, then take the median
median_feeders_per_sub = df.groupby('substation')['feeder'].nunique().median()
print(f"median number of feeders per substation: {median_feeders_per_sub}")
print(f"avarage number of feeders per substation: {df.groupby('substation')['feeder'].nunique().mean()}")

number of feeders: 55
number of substations: 14
median number of feeders per substation: 4.0
avarage number of feeders per substation: 3.9285714285714284
avarage number of feeders per substation: 3.9285714285714284


In [9]:
from IPython.display import display, Markdown
import numpy as np

def print_summary_statistics():
    # Get all unique substations
    substations = df_substations['substation'].unique()
    for sub in substations:
        # Get all feeders for this substation
        feeders_in_sub = [f for f in feeder_dfs if feeder_dfs[f]['substation'].iloc[0] == sub]
        # Combine all data for this substation
        sub_df = pd.concat([feeder_dfs[f] for f in feeders_in_sub], ignore_index=True)
        sub_name = feeder_dfs[feeders_in_sub[0]]['substation_name'].iloc[0] if feeders_in_sub else sub
        display(Markdown(f"### Substation {sub} ({sub_name})"))
        if not sub_df.empty:
            stats = sub_df['consumption'].agg(['min', 'max', 'median', 'mean'])
            display(Markdown(f"**Substation summary:** min: {stats['min']:.2f}, max: {stats['max']:.2f}, median: {stats['median']:.2f}, mean: {stats['mean']:.2f}"))
        else:
            display(Markdown("No data for this substation."))
        for feeder in feeders_in_sub:
            f_df = feeder_dfs[feeder]
            if not f_df.empty:
                f_stats = f_df['consumption'].agg(['min', 'max', 'median', 'mean'])
                display(Markdown(f"- **Feeder {feeder}:** min: {f_stats['min']:.2f}, max: {f_stats['max']:.2f}, median: {f_stats['median']:.2f}, mean: {f_stats['mean']:.2f}"))
            else:
                display(Markdown(f"- **Feeder {feeder}:** No data."))
        display(Markdown("---"))

# Call the function to print the summary
print_summary_statistics()

### Substation A (Estación Carlos Antonio López)

**Substation summary:** min: 1.00, max: 825.00, median: 60.00, mean: 63.13

- **Feeder A1:** min: 1.00, max: 178.00, median: 67.00, mean: 69.67

- **Feeder A2:** min: 4.00, max: 825.00, median: 55.00, mean: 56.71

---

...

- **Feeder N1:** min: 2.00, max: 398.00, median: 185.00, mean: 187.06

- **Feeder N2:** min: 1.00, max: 886.00, median: 126.00, mean: 129.98

- **Feeder N3:** min: 2.00, max: 897.00, median: 128.00, mean: 131.82

- **Feeder N4:** min: 5.00, max: 262.00, median: 97.00, mean: 100.35

---